In [1]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm
import multiprocessing as mp
from transformers import get_linear_schedule_with_warmup

In [2]:
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 1#5
PL_EPOCHS=1#8
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
SEEDS = [42, 123]

In [3]:
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [4]:
df = pd.read_csv(train_path)
df['rule']= df['rule'].str.lower().str.strip()
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

In [5]:
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [6]:
test_df = pd.read_csv(test_path)
test_df['rule']= test_df.rule.str.lower().str.strip()

augmented_train = add_data(df)
augmented_test = add_data(test_df)

augmented_texts =  df.text.tolist()+augmented_train[0] + augmented_test[0]
augmented_labels =  df.label.tolist()+augmented_train[1] + augmented_test[1]

augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before:{augmented_df.shape}')
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',
    'label': 'mean'
})
print('After:',augmented_df.shape)
augmented_df['rule']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body']= augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

rule_map= {i:j for j,i in enumerate(augmented_df.rule.unique())}
augmented_df['rule_id']= augmented_df.rule.map(rule_map)

augmented_df.head()

Before:(10185, 2)
After: (1875, 2)


,text,label,rule,body,rule_id
0,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...","\n\nIf you have some free time on your hands, ...",0
1,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\n\nplease visit http://www.shifadental.net/te...,0
2,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n\nSD | [ English Stream 1 Arsenal vs Tottenh...,0
3,"no advertising: spam, referral links, unsolici...",0.0,"no advertising: spam, referral links, unsolici...",\n**HD** ENG [ 1080P HD Amazing] :- [USTREAM E...,0
4,"no advertising: spam, referral links, unsolici...",1.0,"no advertising: spam, referral links, unsolici...",\nFree http://forums.airdroid.com/viewtopic.ph...,0


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [8]:
unlabelled= pd.read_csv('/kaggle/input/jigsaw-unlabelled-14b/sampled_unlabelled_100k_with_predictions.csv')
unlabelled['rule']= unlabelled['rule'].str.lower().str.strip()
unlabelled['text']= unlabelled['rule']+ ' [SEP] '+ unlabelled['body']
unlabelled['rule_id']= unlabelled.rule.map(rule_map)

pseudo_rules = unlabelled.dropna(subset=['rule_id']).rule.unique().tolist()
print(f'Rules in pseudo data: {pseudo_rules}')
print(f'Number of pseudo rules: {len(pseudo_rules)}')

all_rules = augmented_df.rule.unique().tolist()
non_pseudo_rules = [r for r in all_rules if r not in pseudo_rules]
print(f'Rules without pseudo data: {non_pseudo_rules}')

Rules in pseudo data: ['no legal advice: do not offer or request legal advice.', 'no advertising: spam, referral links, unsolicited advertising, and promotional content are not allowed.']
Number of pseudo rules: 2
Rules without pseudo data: []


In [9]:
# unlabelled_taken.sample(frac=.7).groupby(['rule_id','label']).agg('count')

In [22]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    for seed in SEEDS:
        train_data, val_data = train_test_split(
            augmented_df, 
            test_size=0.2, 
            stratify=augmented_df["rule"], 
            random_state=seed
        )
        unlabelled['label']= unlabelled.rule_violation.round(0)
        unlabelled= unlabelled.query('text not in @augmented_df.text')
        
        # remove test queries especially
        temp_test_data= test_df["rule"].str.lower().str.strip() + " [SEP] " + test_df["body"]
        unlabelled= unlabelled.query('text not in @temp_test_data')
        print(unlabelled.shape)

        confident_positives = unlabelled[unlabelled['rule_violation'] >= 0.85]
        confident_negatives = unlabelled[unlabelled['rule_violation'] <= 0.1]
        
        n_positives = len(confident_positives)
        n_negatives_to_sample = int(n_positives * 1.1)
        
        sampled_negatives = confident_negatives.sample(
          n=min(n_negatives_to_sample, len(confident_negatives)),
          random_state=seed
        )
        
        unlabelled_taken = pd.concat([confident_positives, sampled_negatives], ignore_index=True)
        unlabelled_taken= unlabelled_taken.sample(frac=.7, random_state=seed)
        print(f'Seed {seed} - Pseudo: {len(confident_positives)} pos (>=0.85), {len(sampled_negatives)} neg (<=0.1), total={len(unlabelled_taken)}')
        # unlabelled_taken= unlabelled.sample(frac=.7,random_state=seed)
        
        train_data.to_csv(f'fixed_train_split_seed_{seed}.csv', index=False)
        val_data.to_csv(f'fixed_val_split_seed_{seed}.csv', index=False)
        unlabelled_taken.to_csv(f'fixed_pseudo_seed_{seed}.csv', index=False)
        print(f'Seed {seed} splits saved: train={len(train_data)}, val={len(val_data)}, pseudo={len(unlabelled_taken)}')
    
    import json
    with open('pseudo_rules.json', 'w') as f:
        json.dump({'pseudo_rules': pseudo_rules, 'non_pseudo_rules': non_pseudo_rules}, f)
    print(f'Saved pseudo_rules.json with {len(pseudo_rules)} pseudo rules and {len(non_pseudo_rules)} non-pseudo rules')

(101105, 10)
Seed 42 - Pseudo: 10734 pos (>=0.85), 11807 neg (<=0.1), total=15779
Seed 42 splits saved: train=1500, val=375, pseudo=15779
(101105, 10)
Seed 123 - Pseudo: 10734 pos (>=0.85), 11807 neg (<=0.1), total=15779
Seed 123 splits saved: train=1500, val=375, pseudo=15779
Saved pseudo_rules.json with 2 pseudo rules and 0 non-pseudo rules


In [23]:
class JigsawDataset(Dataset):
    def __init__(self, texts, labels, rule_ids, tokenizer, max_len, weights=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids
        self.weights = weights if weights is not None else [1.0]*len(texts)

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids']= torch.tensor(self.rule_ids[idx])
        item['weights'] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item

In [24]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        logits = self.out(self.drop(pooled)).squeeze(1)
        return logits, pooled

In [25]:
def apply_mixup_pseudo_rules(pooled_orig, labels_orig, pooled_pseudo, labels_pseudo, lam):
    batch_size = min(len(pooled_orig), len(pooled_pseudo))
    
    orig_indices = torch.randperm(len(pooled_orig))[:batch_size]
    pseudo_indices = torch.randperm(len(pooled_pseudo))[:batch_size]
    
    mixed_pooled = lam * pooled_orig[orig_indices] + (1 - lam) * pooled_pseudo[pseudo_indices]
    mixed_labels = lam * labels_orig[orig_indices] + (1 - lam) * labels_pseudo[pseudo_indices]
    
    unmixed_orig_indices = torch.tensor([i for i in range(len(pooled_orig)) if i not in orig_indices.tolist()], device=pooled_orig.device)
    
    if len(unmixed_orig_indices) > 0:
        final_pooled = torch.cat([mixed_pooled, pooled_orig[unmixed_orig_indices]], dim=0)
        final_labels = torch.cat([mixed_labels, labels_orig[unmixed_orig_indices]], dim=0)
    else:
        final_pooled = mixed_pooled
        final_labels = mixed_labels
    
    return final_pooled, final_labels

def train_one_epoch_with_mixup(model, orig_loader, pseudo_loader, optimizer, scheduler, device, mixup_prob=0.3):
    model.train()
    total_loss = 0
    num_batches = 0
    
    pseudo_iter = iter(pseudo_loader)
    
    for orig_batch in tqdm(orig_loader, desc='Training'):
        optimizer.zero_grad()
        
        orig_ids = orig_batch["input_ids"].to(device)
        orig_mask = orig_batch["attention_mask"].to(device)
        orig_labels = orig_batch["labels"].to(device)
        
        logits_orig, pooled_orig = model(orig_ids, orig_mask)
        
        if torch.rand(1).item() < mixup_prob:
            try:
                pseudo_batch = next(pseudo_iter)
            except StopIteration:
                pseudo_iter = iter(pseudo_loader)
                pseudo_batch = next(pseudo_iter)
            
            pseudo_ids = pseudo_batch["input_ids"].to(device)
            pseudo_mask = pseudo_batch["attention_mask"].to(device)
            pseudo_labels = pseudo_batch["labels"].to(device)
            
            with torch.no_grad():
                _, pooled_pseudo = model(pseudo_ids, pseudo_mask)
            
            lam = np.random.beta(1, 1)
            mixed_pooled, mixed_labels = apply_mixup_pseudo_rules(
                pooled_orig, orig_labels, pooled_pseudo, pseudo_labels, lam
            )
            
            logits = model.out(model.drop(mixed_pooled)).squeeze(1)
            loss = nn.BCEWithLogitsLoss()(logits, mixed_labels)
        else:
            loss = nn.BCEWithLogitsLoss()(logits_orig, orig_labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def train_one_epoch_no_mixup(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        logits, _ = model(input_ids, mask)
        
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [26]:
def validate(model, loader, device):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            rule_ids = batch["rule_ids"]
            
            logits, _ = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

In [27]:
def train_model_seed_no_pseudo(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed} NO-PSEUDO] Training on {device}")
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    
    train_ds = JigsawDataset(
        train_data['text'].tolist(), 
        train_data['label'].tolist(), 
        train_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    val_ds = JigsawDataset(
        val_data['text'].tolist(), 
        val_data['label'].tolist(), 
        val_data['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = EPOCHS * len(train_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(EPOCHS):
        print(f"[Seed {seed} NO-PSEUDO] Epoch {epoch+1}/{EPOCHS}")
        loss = train_one_epoch_no_mixup(model, train_loader, optimizer, scheduler, device)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed} NO-PSEUDO] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_no_pseudo_seed_{seed}.bin")
    
    print(f"[Seed {seed} NO-PSEUDO] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_no_pseudo_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

def train_model_seed_with_pseudo(seed, gpu_id):
    import random
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    device = torch.device(f"cuda:{gpu_id}")
    print(f"[Seed {seed} WITH-PSEUDO] Training on {device}")
    
    import json
    with open('pseudo_rules.json', 'r') as f:
        rules_info = json.load(f)
    pseudo_rules = rules_info['pseudo_rules']
    
    train_data = pd.read_csv(f'fixed_train_split_seed_{seed}.csv')
    val_data = pd.read_csv(f'fixed_val_split_seed_{seed}.csv')
    unlabelled_taken = pd.read_csv(f'fixed_pseudo_seed_{seed}.csv')
    
    train_data_pseudo_rules = train_data[train_data['rule'].isin(pseudo_rules)].copy()
    val_data_pseudo_rules = val_data[val_data['rule'].isin(pseudo_rules)].copy()
    
    print(f"[Seed {seed} WITH-PSEUDO] Training on {len(train_data_pseudo_rules)} original samples from {len(pseudo_rules)} pseudo rules: {pseudo_rules}")
    print(f"[Seed {seed} WITH-PSEUDO] Using {len(unlabelled_taken)} pseudo samples for mixup")
    
    orig_ds = JigsawDataset(
        train_data_pseudo_rules['text'].tolist(), 
        train_data_pseudo_rules['label'].tolist(), 
        train_data_pseudo_rules['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    pseudo_ds = JigsawDataset(
        unlabelled_taken['text'].tolist(), 
        unlabelled_taken['label'].tolist(), 
        unlabelled_taken['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    val_ds = JigsawDataset(
        val_data_pseudo_rules['text'].tolist(), 
        val_data_pseudo_rules['label'].tolist(), 
        val_data_pseudo_rules['rule_id'].tolist(), 
        tokenizer, MAX_LEN
    )
    
    orig_loader = DataLoader(orig_ds, batch_size=BATCH_SIZE, shuffle=True)
    pseudo_loader = DataLoader(pseudo_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
    model = JigsawModel(MODEL_PATH).to(device)
    for name, param in model.named_parameters():
        if name.startswith('base.embedding'):
            param.requires_grad = False
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
    total_steps = PL_EPOCHS * len(orig_loader)
    warmup_steps = int(0.1 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    best_auc = 0
    best_loss= None
    for epoch in range(PL_EPOCHS):
        print(f"[Seed {seed} WITH-PSEUDO] Epoch {epoch+1}/{PL_EPOCHS}")
        loss = train_one_epoch_with_mixup(model, orig_loader, pseudo_loader, optimizer, scheduler, device, mixup_prob=0.3)
        val_auc, val_loss, val_preds = validate(model, val_loader, device)
        
        print(f"[Seed {seed} WITH-PSEUDO] Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
        if val_auc > best_auc:
            best_auc = val_auc
            best_loss= val_loss
            torch.save(model.state_dict(), f"model_with_pseudo_seed_{seed}.bin")
    
    print(f"[Seed {seed} WITH-PSEUDO] Best validation AUC: {best_auc:.4f}")
    import json
    with open(f'results_with_pseudo_seed_{seed}.json', 'w') as f:
        json.dump({'seed': seed, 'best_auc': best_auc, 'best_loss': best_loss}, f)

    return seed, best_auc

In [28]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
      import torch.multiprocessing as mp
      mp.set_start_method('fork', force=True)

      # Train NO-PSEUDO models first
      print("=== Training NO-PSEUDO models ===")
      processes = []
      for idx, seed in enumerate(SEEDS):
         gpu_id = idx % torch.cuda.device_count()
         p = mp.Process(target=train_model_seed_no_pseudo, args=(seed, gpu_id))
         p.start()
         processes.append(p)

      for p in processes:
         p.join()

      # Clear memory
      torch.cuda.empty_cache()
      import gc
      gc.collect()

      # Train WITH-PSEUDO models next
      print("\n=== Training WITH-PSEUDO models ===")
      processes = []
      for idx, seed in enumerate(SEEDS):
         gpu_id = idx % torch.cuda.device_count()
         p = mp.Process(target=train_model_seed_with_pseudo, args=(seed, gpu_id))
         p.start()
         processes.append(p)

      for p in processes:
         p.join()

      import json
      results_no_pseudo = []
      results_with_pseudo = []

      for seed in SEEDS:
        with open(f'results_no_pseudo_seed_{seed}.json', 'r') as f:
            results_no_pseudo.append(json.load(f))
        with open(f'results_with_pseudo_seed_{seed}.json', 'r') as f:
            results_with_pseudo.append(json.load(f))

      aucs_no = [r['best_auc'] for r in results_no_pseudo]
      losses_no = [r['best_loss'] for r in results_no_pseudo]
      aucs_with = [r['best_auc'] for r in results_with_pseudo]
      losses_with = [r['best_loss'] for r in results_with_pseudo]

      print("\n=== NO PSEUDO (for non-pseudo rules) ===")
      print(f"AUC: {np.mean(aucs_no):.4f} ± {np.std(aucs_no):.4f}")
      print(f"Loss: {np.mean(losses_no):.4f} ± {np.std(losses_no):.4f}")

      print("\n=== WITH PSEUDO (for pseudo rules) ===")
      print(f"AUC: {np.mean(aucs_with):.4f} ± {np.std(aucs_with):.4f}")
      print(f"Loss: {np.mean(losses_with):.4f} ± {np.std(losses_with):.4f}")

      print("\nAll 4 models trained!")


=== Training NO-PSEUDO models ===
[Seed 42 NO-PSEUDO] Training on cuda:0
[Seed 123 NO-PSEUDO] Training on cuda:1


2025-10-07 14:40:46.729926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-07 14:40:46.729926: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759848046.972528      90 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759848046.972519      91 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759848047.038731      90 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1759848047.038730      91 cuda_blas.cc:1

[Seed 42 NO-PSEUDO] Epoch 1/1[Seed 123 NO-PSEUDO] Epoch 1/1



Training: 100%|██████████| 47/47 [00:36<00:00,  1.29it/s]


[Seed 42 NO-PSEUDO] Loss: 0.6918, Val Loss: 0.6849, Val AUC: 0.6902
[Seed 123 NO-PSEUDO] Loss: 0.6811, Val Loss: 0.6605, Val AUC: 0.7386
[Seed 42 NO-PSEUDO] Best validation AUC: 0.6902
[Seed 123 NO-PSEUDO] Best validation AUC: 0.7386

=== Training WITH-PSEUDO models ===
[Seed 42 WITH-PSEUDO] Training on cuda:0
[Seed 123 WITH-PSEUDO] Training on cuda:1
[Seed 42 WITH-PSEUDO] Training on 1500 original samples from 2 pseudo rules: ['no legal advice: do not offer or request legal advice.', 'no advertising: spam, referral links, unsolicited advertising, and promotional content are not allowed.']
[Seed 42 WITH-PSEUDO] Using 15779 pseudo samples for mixup[Seed 123 WITH-PSEUDO] Training on 1500 original samples from 2 pseudo rules: ['no legal advice: do not offer or request legal advice.', 'no advertising: spam, referral links, unsolicited advertising, and promotional content are not allowed.']

[Seed 123 WITH-PSEUDO] Using 15779 pseudo samples for mixup


...

[Seed 123 WITH-PSEUDO] Epoch 1/1


Training:   0%|          | 0/47 [00:00<?, ?it/s]

[Seed 42 WITH-PSEUDO] Epoch 1/1


Training: 100%|██████████| 47/47 [00:41<00:00,  1.13it/s]


[Seed 123 WITH-PSEUDO] Loss: 0.6851, Val Loss: 0.6670, Val AUC: 0.7401
[Seed 42 WITH-PSEUDO] Loss: 0.6954, Val Loss: 0.6858, Val AUC: 0.6813
[Seed 123 WITH-PSEUDO] Best validation AUC: 0.7401
[Seed 42 WITH-PSEUDO] Best validation AUC: 0.6813

=== NO PSEUDO (for non-pseudo rules) ===
AUC: 0.7144 ± 0.0242
Loss: 0.6727 ± 0.0122

=== WITH PSEUDO (for pseudo rules) ===
AUC: 0.7107 ± 0.0294
Loss: 0.6764 ± 0.0094

All 4 models trained!


In [29]:
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    import json
    with open('pseudo_rules.json', 'r') as f:
        rules_info = json.load(f)
    pseudo_rules = rules_info['pseudo_rules']
    
    df_test = pd.read_csv(test_path)
    df_test['rule'] = df_test['rule'].str.lower().str.strip()
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    df_test_pseudo = df_test[df_test['rule'].isin(pseudo_rules)].copy()
    df_test_no_pseudo = df_test[~df_test['rule'].isin(pseudo_rules)].copy()
    
    print(f'Test samples with pseudo rules: {len(df_test_pseudo)}')
    print(f'Test samples without pseudo rules: {len(df_test_no_pseudo)}')
    
    device = torch.device("cuda:0")
    
    if len(df_test_no_pseudo) > 0:
        test_ds_no_pseudo = JigsawDataset(
            df_test_no_pseudo['text'].tolist(), 
            [0]*len(df_test_no_pseudo), 
            [0]*len(df_test_no_pseudo), 
            tokenizer, MAX_LEN
        )
        test_loader_no_pseudo = DataLoader(test_ds_no_pseudo, batch_size=BATCH_SIZE)
        
        all_preds_no_pseudo = []
        for seed in SEEDS:
            model = JigsawModel(MODEL_PATH).to(device)
            model.load_state_dict(torch.load(f"model_no_pseudo_seed_{seed}.bin", map_location=device))
            model.eval()
            
            test_preds = []
            with torch.no_grad():
                for batch in tqdm(test_loader_no_pseudo, desc=f"Inference NO-PSEUDO seed {seed}"):
                    ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    logits, _ = model(ids, mask)
                    test_preds.extend(torch.sigmoid(logits).cpu().numpy())
            
            all_preds_no_pseudo.append(test_preds)
        
        ensemble_preds_no_pseudo = np.mean(all_preds_no_pseudo, axis=0)
        df_test_no_pseudo['rule_violation'] = ensemble_preds_no_pseudo
    
    if len(df_test_pseudo) > 0:
        test_ds_pseudo = JigsawDataset(
            df_test_pseudo['text'].tolist(), 
            [0]*len(df_test_pseudo), 
            [0]*len(df_test_pseudo), 
            tokenizer, MAX_LEN
        )
        test_loader_pseudo = DataLoader(test_ds_pseudo, batch_size=BATCH_SIZE)
        
        all_preds_pseudo = []
        for seed in SEEDS:
            model = JigsawModel(MODEL_PATH).to(device)
            model.load_state_dict(torch.load(f"model_with_pseudo_seed_{seed}.bin", map_location=device))
            model.eval()
            
            test_preds = []
            with torch.no_grad():
                for batch in tqdm(test_loader_pseudo, desc=f"Inference WITH-PSEUDO seed {seed}"):
                    ids = batch['input_ids'].to(device)
                    mask = batch['attention_mask'].to(device)
                    logits, _ = model(ids, mask)
                    test_preds.extend(torch.sigmoid(logits).cpu().numpy())
            
            all_preds_pseudo.append(test_preds)
        
        ensemble_preds_pseudo = np.mean(all_preds_pseudo, axis=0)
        df_test_pseudo['rule_violation'] = ensemble_preds_pseudo
    
    df_test_final = pd.concat([df_test_no_pseudo, df_test_pseudo]).sort_index()
    
    sample = pd.read_csv(sample_sub_path)
    sample["rule_violation"] = df_test_final["rule_violation"].values
    sample.to_csv("submission.csv", index=False)
    print(f"Ensembled {len(SEEDS)} models per type (total 4 models)")
else:
    !touch submission.csv
    
!head -n 4 submission.csv

Test samples with pseudo rules: 10
Test samples without pseudo rules: 0


2025-10-07 14:42:43.839224: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759848163.862104      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759848163.869124      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Inference WITH-PSEUDO seed 123: 100%|██████████| 1/1 [00:00<00:00,  4.22it/s]

Ensembled 2 models per type (total 4 models)
row_id,rule_violation
2029,0.46505332
2030,0.51768386
2031,0.5050589


In [30]:
# 4 models total: 2 seeds × 2 types (with/without pseudo)
# NO-PSEUDO models: trained on ALL rules with original data only, used for non-pseudo rules at inference
# WITH-PSEUDO models: trained on ONLY pseudo rules with original + pseudo data mixup, used for pseudo rules at inference
# Inference routes test samples to appropriate model based on their rule

In [31]:
#ablation
#start                            .8765 3rd epoch .4731 val loss
#all pseudo 
#random sample pseudo 2k          .8871 3nd epoch, .4731 val loss
#random sample pseudo 4k          .8977 .8761 2.5th epoch, .4133 .4663 val loss
#random sample pseudo 8k          .8979 .8932 3.5th epochs, .4223 .4409
#random sample pseudo 16k         .9082 .8904 3th epoch  .4356 .4363 
#random sample pseudo 32k         .8893 .9104 2.5th epoch  .4494 .4286 ***
#random sample pseudo 64k         .8913 3th epoch  .4298
#random sample pseudo 100k        .8913 3th epoch  .4298

# weightitng starts 
#.3 weights

# good options
#. 5 weight, hard labels, remove .1-.85, balance to have same, 20k points taken, dontoverwhelm with pseudo data, for each rule 10k points, so at best have 20k points(10k pairs), at best we have 10k points, so use them with diff wights .5-

In [32]:
# FINAL CLEAN ARCHITECTURE:
# 
# NO-PSEUDO MODEL:
# - Trains on ALL rules with original data only
# - Used for inference on non-pseudo rules
# - Standard training, no mixup
#
# WITH-PSEUDO MODEL:
# - Trains ONLY on pseudo rules (filters train data to rules with pseudo data available)
# - Uses original data from pseudo rules + pseudo data for mixup
# - 40% of batches: Mixup between original and pseudo (both from pseudo rules)
# - 60% of batches: Pure original data from pseudo rules
# - Used for inference on pseudo rules only
#
# MIXUP STRATEGY:
# - Both samples in mixup are from pseudo rules (can be different rules)
# - No requirement for exact rule match
# - Mixup at embedding level after last hidden layer
# - Beta(0.3, 0.3) for lambda, soft labels from pseudo data
# - Epoch size = original data from pseudo rules only (subset of full training data)

In [33]:
#DONE if mixup works let it work with balanced data, hrdlbls, .85 conf, no wts, with mixup .3 prob, 1 mxp-param,8 epochs
#TODO next mixup ideas, this type of mixup may not be working